# dyadmocap-temporal-mapper: summary of all dyads

This notebook builds the **Temporal Mapper graph** of every dyad in both conversations, shows the graphs, and then summarises the main **graph and cycle metrics** in one table. It does no statistics.

It uses **simulated** data, because the study's recordings are not public. The code is the same code that analysed the study data; only the input files differ. The numbers describe the simulation only; they are **not findings** about real conversations.

Each step is explained in `dyadmocap_one_dyad.ipynb`; the full analysis with statistics is in `dyadmocap_walkthrough.ipynb`.

Temporal Mapper is an external dependency ([tmapper-py](https://github.com/Multiscale-Complex-Systems-Lab/tmapper-py)), installed by `environment.yml`.

## 0. Setup

Run this notebook from the `python/notebooks` folder, in the `dyadmocap-tm` environment (see the README). The first cell makes the package importable and checks that Temporal Mapper is installed.

In [ ]:
import sys, os, json, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Image, Video

PYTHON_DIR = Path.cwd().parent            # .../python
REPO = PYTHON_DIR.parent
sys.path.insert(0, str(PYTHON_DIR))

import tmapper                              # external dependency
from dyadmocap import dymo_settings as S
print("Temporal Mapper (tmapper-py) found:", Path(tmapper.__file__).parent)
print("Dataset index :", S.DATA_INDEX)
print("Results folder:", S.PY_RESULTS_DIR)

def show(path, width=750):
    # show a saved PNG figure in the notebook
    display(Image(filename=str(path), width=width))

### The data

`data/simulated/simulate_dyad.py` writes 15 simulated dyads in the file layout of the study's motion-capture export (one tab-delimited file per person and conversation, 42 markers). Two dyads have a conversation that is too short **on purpose**, so the inclusion rule (below) keeps **13**. The files are written once (1–3 minutes, about 0.2 GB) with fixed seeds, so every run gives identical files.

To analyse your own data instead, copy `dymo_config.example.ini` to `dymo_config.ini` and point `data_index` at an index file of your recordings (the format is described in that file).

In [ ]:
if not Path(S.DATA_INDEX).is_file():
    print("Writing the simulated dyads (once)...")
    subprocess.run([sys.executable, str(REPO / "data" / "simulated" / "simulate_dyad.py")], check=True)

index = S.load_index()
dyads = index["dyads"]
n_short = sum(d.get("excluded_by_design", False) for d in dyads.values())
print("%d dyads in the index; %d too short by design" % (len(dyads), n_short))
EXAMPLE = "simdyad01"                       # the example dyad of Part A
TAG = " (simulated data)"

## 1. Loading

Each recording is trimmed (first 25 s and last 5 s dropped), a common centred window is chosen for the two conversations, short gaps are filled, and the first 180 s are analysed. A dyad is included only if every recording keeps at least 180 s.

In [ ]:
from dyadmocap.dymo_loading import find_times, load_group

times = find_times()
data, fs = {}, None
for group in S.GROUPS:
    data[group], _, fs, n_keep = load_group(group, times)
    print("%s: %d dyads included, %d samples per recording (%.0f s at %.0f Hz)"
          % (group, len(data[group]), n_keep, n_keep / fs, fs))
print("Excluded:", sorted(set(dyads) - set(data["GTK"])))

## 2. Temporal Mapper graph of every dyad

For each dyad and conversation: 11 joint angles per person → velocity (22 columns) → the four behavioural states (median rule) → the Temporal Mapper graph (k = 5, dmap = 3, texclude = 3 samples). Each node is coloured by the state most of its moments carry: Both Inactive black, Both Active green, Person 1 Active blue, Person 2 Active red.

In [ ]:
import matplotlib.pyplot as plt
from dyadmocap import dymo_figures as F
from dyadmocap.dymo_angles import joint_angles, velocity
from dyadmocap.dymo_categories import normalize_median, categorize
from dyadmocap.dymo_graph import build_graph_with_samples

N_ANG = 11
GDIR = Path(S.PY_RESULTS_DIR) / "summary"
included = sorted(data["GTK"])
tm = {}
for d in included:
    for group in S.GROUPS:
        p = data[group][d]
        a1, _ = joint_angles(p[0]); a2, _ = joint_angles(p[1])
        v = velocity(a1, a2, fs)
        r1, _ = normalize_median(v[:, :N_ANG]); r2, _ = normalize_median(v[:, N_ANG:])
        colorvar = categorize(r1, r2)["colorvar"]
        g_simp, members, nodesize, g = build_graph_with_samples(v, S.K, S.DMAP, S.TEXCLUDE)
        tm[d, group] = dict(g_simp=g_simp, members=members, nodesize=np.asarray(nodesize), g=g, colorvar=colorvar)
        F.four_color_graph(g_simp, colorvar, members, "%s %s%s" % (group, d, TAG), GDIR, "%s_%s_FourColorGraph" % (d, group))
    print("%s: %d / %d nodes (casual / conflict)"
          % (d, tm[d, "GTK"]["g_simp"].number_of_nodes(), tm[d, "MAN"]["g_simp"].number_of_nodes()))

# all graphs on one page: casual chat left, conflict right
fig, axes = plt.subplots(len(included), 2, figsize=(10, 4.2 * len(included)))
for r, d in enumerate(included):
    for c, (group, name) in enumerate([("GTK", "Casual chat"), ("MAN", "Conflict")]):
        ax = axes[r, c]
        ax.imshow(plt.imread(GDIR / ("%s_%s_FourColorGraph.png" % (d, group))))
        ax.set_axis_off()
        ax.set_title("%s, %s" % (d, name), fontsize=10)
plt.tight_layout()
plt.show()

## 3. Graph and cycle metrics of every dyad

* **Number of nodes**: how many movement states the graph has.
* **Node size**: the mean number of time points per node.
* **Edge density**: the share of possible transitions present.
* **Mean path distance**: the average number of steps from a node to all others (and from all others to it), for the whole graph and for the nodes of each state.
* **Cycles**: loops back to the same node: how many, their mean length, and how many states a loop passes through (Shannon entropy of its states; 0 = one state only).

One row per dyad and conversation. No statistics.

In [ ]:
from dyadmocap.dymo_graph_metrics import whole_graph_metrics, node_labels
from dyadmocap.dymo_cycles import find_cycles, cycle_data, whole_cycle_metrics
from dyadmocap.dymo_node_distance import compute_dsimp
from dyadmocap.dymo_hubness import node_closeness_by_category

STATE_NAMES = ["Both Inactive", "Both Active", "Person 1 Active", "Person 2 Active"]

def graph_cycle_metrics(T):
    """The main graph and cycle metrics of one Temporal Mapper graph."""
    W = whole_graph_metrics(T["g_simp"], T["nodesize"])
    labels = node_labels(T["members"], T["colorvar"])
    C = whole_cycle_metrics(cycle_data(find_cycles(T["g_simp"]), labels), T["g_simp"].number_of_nodes())
    D = compute_dsimp(T["g"], T["members"])
    mpd_all = node_closeness_by_category(D, np.ones(len(labels), dtype=int), 1)[0][0]
    mpd_state = node_closeness_by_category(D, labels)[0]
    row = {"Number of nodes": W["numNodes"],
           "Node size (mean time points)": W["nodesizes"],
           "Edge density": W["density"],
           "Mean path distance": mpd_all}
    row.update({"Mean path distance, " + n: mpd_state[i] for i, n in enumerate(STATE_NAMES)})
    row.update({"Cycle count": C["CycleCount"],
                "Cycle mean length": C["CycleLenMean"],
                "Cycle entropy (mean)": C["meanEntropy"]})
    return row

# whole numbers as they are, others to 4 significant figures
pd.set_option("display.float_format", lambda x: "%d" % x if float(x).is_integer() else "%.4g" % x)

In [ ]:
rows = []
for d in included:
    for group, name in [("GTK", "Casual chat"), ("MAN", "Conflict")]:
        rows.append({"Dyad": d, "Conversation": name, **graph_cycle_metrics(tm[d, group])})
summary = pd.DataFrame(rows).set_index(["Dyad", "Conversation"])
summary.to_csv(GDIR / ("Summary_GraphCycleMetrics_%s.csv" % S.PY_TAG))
summary

## Where everything is

The graphs and the table are in the `summary` folder of the results folder printed in the setup cell.

*Before committing this notebook, clear all outputs (Kernel → Restart Kernel and Clear Outputs of All Cells).*